# 1장 인공지능, 머신러닝, 딥러닝 — 규칙을 짜는 프로그램과 규칙을 배우는 프로그램
『머신러닝과 딥러닝 기초』 실습 노트북

이 노트북에서는 두 가지를 해 봅니다.
1. 사람이 규칙을 직접 적는 **규칙 기반** 프로그램
2. 데이터에서 규칙을 찾는 **머신러닝** 프로그램 (택시 요금 규칙 찾기)

아직 코드를 모두 이해하지 못해도 괜찮습니다. 위에서부터 차례로 실행하며 결과를 확인하세요.

## 1. 규칙 기반 프로그램: 보이스피싱 문자 골라내기
수사 기관을 사칭하며 돈을 옮기라는 문자를 의심하도록, 사람이 규칙을 `if` 문으로 적었습니다.

In [ ]:
def is_phishing(text):
    # 수사 기관 사칭 + 송금 요구가 함께 있으면 의심
    if "검찰" in text and "계좌" in text:
        return True
    return False

messages = [
    "서울중앙지검 검찰 수사관입니다. 안전 계좌로 이체하세요.",
    "서울중앙지검 검.찰 수사관입니다. 안전 계.좌로 이체하세요.",   # 글자를 살짝 바꾼 피싱 문자
    "검찰청 앞 카페에서 만나. 회비는 내 계좌로 보내 줘.",            # 평범한 문자
]
for i, m in enumerate(messages, start=1):
    print(f"{i}번 문자 → 피싱 의심: {is_phishing(m)}")

두 번째 문자는 분명 피싱인데 걸러지지 않았고, 세 번째 평범한 문자는 피싱으로 잘못 걸렸습니다.
글자를 살짝 바꾸는 수법, 사칭 대상을 바꾸는 수법이 나올 때마다 규칙을 늘려야 하고, 규칙을 늘리면 평범한 문자가 잘못 걸리는 일도 늘어납니다.

## 2. 머신러닝: 택시 요금 규칙 찾기
어느 도시의 택시 요금 규칙을 모른 채, 승차 기록(거리, 요금)만 가지고 있다고 합시다.

먼저 연습용 기록을 만듭니다. 실제 규칙은 **기본요금 4,800원 + 1 km당 1,000원**이고, 신호 대기 같은 이유로 생기는 작은 차이(잡음, noise)를 섞었습니다.
이 규칙은 데이터를 만들 때만 쓰고, 학습할 때는 컴퓨터에게 알려 주지 않습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=7)          # 결과 고정
n = 50                                       # 기록 수
km = rng.uniform(1, 20, size=n).round(1)     # 1~20 km 거리
noise = rng.normal(0, 600, size=n)           # 잡음
fare = (1000 * km + 4800 + noise).round(-1)  # 10원 단위

for d, f in zip(km[:8], fare[:8]):
    print(f"{d:5.1f} km  {f:8,.0f} 원")

In [ ]:
plt.scatter(km, fare)
plt.xlabel("distance (km)")
plt.ylabel("fare (won)")
plt.title("Taxi rides")
plt.show()

### 학습: 기록으로 규칙 찾기
사이킷런(scikit-learn)의 `LinearRegression`은 데이터에 가장 잘 맞는 직선을 찾습니다. 어떻게 찾는지는 7·8장에서 배웁니다.

`fit()`이 **학습**입니다.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
# 입력은 2차원 표(50행 1열)로, coef_는 목록이라 [0]으로 꺼낸다
model.fit(km.reshape(-1, 1), fare)   # 학습: 기록으로 규칙 찾기
print(f"기본요금: {model.intercept_:,.0f} 원")
print(f"1 km당 요금: {model.coef_[0]:,.0f} 원")

요금표를 보지 않았는데도 실제 규칙(4,800원, 1,000원)에 아주 가까운 값을 찾았습니다. 조금 다른 것은 기록에 잡음이 섞여 있기 때문입니다.

In [ ]:
xs = np.linspace(0, 21, 100)
plt.scatter(km, fare, label="rides")
plt.plot(xs, model.predict(xs.reshape(-1, 1)), color="red", label="learned rule")
plt.xlabel("distance (km)")
plt.ylabel("fare (won)")
plt.legend()
plt.show()

### 예측: 처음 보는 거리의 요금
학습한 모델은 기록에 없던 거리의 요금도 예측합니다.

In [ ]:
new_km = np.array([[3.0], [12.5], [25.0]])
for d, p in zip(new_km.ravel(), model.predict(new_km)):
    print(f"{d:5.1f} km → 예상 요금 {p:8,.0f} 원")

25 km는 기록(1\~20 km) 밖의 거리입니다. 실제 요금도 직선 규칙을 따른다면 맞겠지만, 장거리 할증 같은 다른 규칙이 있다면 틀릴 수 있습니다.
모델은 **본 적 있는 범위**에서 가장 믿을 만하다는 점을 기억해 둡시다.

---
## 실습 문제

**문제 1.** 다음 문제를 규칙 기반이 알맞은 문제와 머신러닝이 알맞은 문제로 구분하고, 그 이유를 설명하세요.
- (a) 학번 앞 네 자리로 입학 연도 알아내기
- (b) 손으로 쓴 우편번호 숫자 읽기
- (c) 지난 3년의 판매 기록으로 다음 달 우산 판매량 예측하기
- (d) 주민등록번호 뒷자리 첫 숫자로 출생 세기 알아내기

(답을 이 칸 아래에 적으세요.)

**문제 2.** 기록 수 `n`을 50에서 5로 줄여 데이터를 다시 만들고 학습해 보세요. 찾은 기본요금과 1 km당 요금은 원래 규칙(4,800원, 1,000원)과 얼마나 달라지나요? 기록 수가 왜 중요한지 설명하세요.

In [ ]:
# 문제 2 풀이


**문제 3.** 잡음의 크기(표준편차 600)를 10배로 키워 데이터를 다시 만들고 학습해 보세요. 찾은 규칙은 어떻게 달라지나요?

In [ ]:
# 문제 3 풀이


**문제 4.** 최근 1년 사이 나온 인공지능 서비스 하나를 골라 입력과 출력이 무엇인지, 어떤 사회적 문제(편향, 환각, 책임, 저작권·위조, 일자리)와 관련이 있는지 정리하세요.